# Tutorial 2: Model Description

**Sickle Cell Disease Classification — Architecture Details**

This tutorial covers:
1. Loading and saving each model
2. Parameter counting (total & trainable)
3. Input layer compatibility
4. Output layer specifications (activation functions, loss functions)
5. Intermediate layer architecture details

In [35]:
# Install necessary libraries
!pip install -q ultralytics torch torchvision opencv-python-headless matplotlib seaborn scikit-learn pandas numpy albumentations timm kaggle tqdm pyyaml Pillow grad-cam

In [36]:
# Download the dataset from Kaggle and unzip it
!kaggle datasets download -d florencetushabe/sickle-cell-disease-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/florencetushabe/sickle-cell-disease-dataset
License(s): DbCL-1.0
100% 253M/253M [00:12<00:00, 21.9MB/s] 



In [37]:
import os, shutil, random, yaml, time, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (f1_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, confusion_matrix)
from ultralytics import YOLO
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

## Setup Prepare Dataset (from Tutorial 1)

In [38]:
DATA_DIR = "."
os.makedirs("outputs", exist_ok=True)

LABELLED_DIR = os.path.join(DATA_DIR, "Positive", "Labelled")
RAW_DIR      = os.path.join(DATA_DIR, "Positive", "Unlabelled")
CLEAR_DIR    = os.path.join(DATA_DIR, "Negative", "Clear")

# --- Extract YOLO labels by comparing labelled vs unlabelled images ---
LABELS_OUT = "data/processed/labels"
os.makedirs(LABELS_OUT, exist_ok=True)

for f in sorted(os.listdir(LABELLED_DIR)):
    if not f.lower().endswith(('.jpg','.png','.jpeg')):
        continue
    lab_p, raw_p = os.path.join(LABELLED_DIR, f), os.path.join(RAW_DIR, f)
    if not os.path.exists(raw_p):
        continue
    lab, raw = cv2.imread(lab_p), cv2.imread(raw_p)
    if lab is None or raw is None:
        continue
    h, w = lab.shape[:2]
    diff = cv2.absdiff(lab, cv2.resize(raw, (w, h)))
    th = cv2.morphologyEx(
        cv2.threshold(cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY), 40, 255, cv2.THRESH_BINARY)[1],
        cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (3,3)), 2)
    boxes = []
    for c in cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x,y,bw,bh = cv2.boundingRect(c)
        if not (400 < bw*bh < 0.1*w*h): continue
        if max(bw,bh)/(min(bw,bh)+1e-6) > 4: continue
        if not (4 <= len(cv2.approxPolyDP(c, 0.04*cv2.arcLength(c,True), True)) <= 12): continue
        boxes.append([1,(x+bw/2)/w,(y+bh/2)/h,bw/w,bh/h])
    if not boxes:
        continue
    out = os.path.join(LABELS_OUT, f.replace(".jpg",".txt").replace(".png",".txt"))
    with open(out, "w") as fp:
        fp.write("\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c,x,y,w,h in boxes))

# --- Create train / val / test splits ---
YOLO_ROOT = "data/yolo_dataset"
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(YOLO_ROOT, "images", split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_ROOT, "labels", split), exist_ok=True)

pos_labelled_files = sorted([f for f in os.listdir(LABELLED_DIR)
    if f.lower().endswith(('.jpg','.jpeg','.png'))
    and os.path.exists(os.path.join(LABELS_OUT, os.path.splitext(f)[0]+".txt"))])

pos_train, pos_temp = train_test_split(pos_labelled_files, test_size=0.4, random_state=42)
pos_val, pos_test   = train_test_split(pos_temp, test_size=0.5, random_state=42)

for split_name, split_files in [("train",pos_train),("val",pos_val),("test",pos_test)]:
    for fname in split_files:
        s = os.path.join(RAW_DIR, fname)
        if not os.path.exists(s): s = os.path.join(LABELLED_DIR, fname)
        shutil.copy2(s, os.path.join(YOLO_ROOT,"images",split_name,fname))
        lbl = os.path.splitext(fname)[0]+".txt"
        shutil.copy2(os.path.join(LABELS_OUT,lbl), os.path.join(YOLO_ROOT,"labels",split_name,lbl))

neg_files = sorted([f for f in os.listdir(CLEAR_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))])
neg_train, neg_temp = train_test_split(neg_files, test_size=0.4, random_state=42)
neg_val, neg_test   = train_test_split(neg_temp, test_size=0.5, random_state=42)

for split_name, split_files in [("train",neg_train),("val",neg_val),("test",neg_test)]:
    for fname in split_files:
        shutil.copy2(os.path.join(CLEAR_DIR,fname), os.path.join(YOLO_ROOT,"images",split_name,f"neg_{fname}"))
        lbl = f"neg_{os.path.splitext(fname)[0]}.txt"
        with open(os.path.join(YOLO_ROOT,"labels",split_name,lbl),"w") as f:
            f.write("0 0.500000 0.500000 1.000000 1.000000\n")

yaml_path = os.path.join(YOLO_ROOT, "dataset.yaml")
config = {"path":os.path.abspath(YOLO_ROOT),"train":"images/train","val":"images/val",
          "test":"images/test","nc":2,"names":["normal","sickle"]}
with open(yaml_path,"w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Dataset ready: {len(pos_train)+len(neg_train)} train | "
      f"{len(pos_val)+len(neg_val)} val | {len(pos_test)+len(neg_test)} test")

Dataset ready: 339 train | 113 val | 114 test


## 1. YOLOv8n (Nano)

YOLOv8 is a single-stage object detector that predicts bounding boxes and class probabilities in one forward pass. The **nano** variant balances speed and accuracy for resource-constrained environments.

| Property | Value |
|----------|-------|
| Input size | 640 × 640 × 3 (RGB) |
| Backbone | CSPDarknet (Cross Stage Partial) |
| Neck | PANet (Path Aggregation Network) |
| Head | Decoupled detection head (classification + regression) |
| Output | Per-anchor: `[x, y, w, h, objectness, class_0, class_1]` |
| Activation (hidden) | SiLU (Sigmoid Linear Unit) |
| Activation (output) | Sigmoid (objectness & class probabilities) |
| Loss | Box: CIoU loss · Class: BCE loss · DFL (Distribution Focal Loss) |

In [39]:
model_v8 = YOLO("yolov8n.pt")

print("YOLOv8n Architecture Summary")
print("=" * 60)
model_v8.info(verbose=True)

YOLOv8n Architecture Summary
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8575488)

In [40]:
print("\nYOLOv8n — Saving and Loading")
print("=" * 60)

# Save (YOLO saves best.pt automatically during training)
save_path = "weights/yolov8n_demo.pt"
os.makedirs("weights", exist_ok=True)
import shutil as _sh
_sh.copy2("yolov8n.pt", save_path)
print(f"Saved to: {save_path}  ({os.path.getsize(save_path)/1e6:.1f} MB)")

# Load
loaded = YOLO(save_path)
print(f"Loaded from: {save_path}")
print(f"Classes: {loaded.names}")


YOLOv8n — Saving and Loading
Saved to: weights/yolov8n_demo.pt  (6.5 MB)
Loaded from: weights/yolov8n_demo.pt
Classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dinin

## 2. YOLO12n (Nano)

YOLO12 is a newer YOLO variant with an updated architecture that reduces parameters and FLOPs while maintaining comparable detection performance.

| Property | Value |
|----------|-------|
| Input size | 640 × 640 × 3 (RGB) |
| Backbone | Improved CSP with attention mechanisms |
| Neck | Enhanced feature pyramid |
| Head | Decoupled detection head |
| Output | Same format as YOLOv8 |
| Activation (hidden) | SiLU |
| Activation (output) | Sigmoid |
| Loss | Box: CIoU · Class: BCE · DFL |

In [41]:
model_v12 = YOLO("yolo12n.pt")

print("YOLO12n Architecture Summary")
print("=" * 60)
model_v12.info(verbose=True)

YOLO12n Architecture Summary
YOLOv12n summary: 272 layers, 2,603,056 parameters, 0 gradients, 6.7 GFLOPs


(272, 2603056, 0, 6.6535552)

## 3. SW-ViT (Swin Transformer Base)

The Swin Transformer uses **shifted windows** for self-attention, enabling efficient processing of high-resolution images with linear computational complexity.

| Property | Value |
|----------|-------|
| Input size | 224 × 224 × 3 (RGB) |
| Patch size | 4 × 4 |
| Window size | 7 × 7 |
| Stages | 4 (embed dims: 128 → 256 → 512 → 1024) |
| Attention heads | [4, 8, 16, 32] per stage |
| Activation (hidden) | GELU |
| Activation (output) | Softmax (2-class) |
| Loss | CrossEntropyLoss |
| Classifier head | Linear(1024 → 2) |

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_swvit = timm.create_model(
    'swin_base_patch4_window7_224',
    pretrained=True,
    num_classes=2
)
model_swvit = model_swvit.to(device)

total_params     = sum(p.numel() for p in model_swvit.parameters())
trainable_params = sum(p.numel() for p in model_swvit.parameters() if p.requires_grad)

print("Swin Transformer — Base")
print("=" * 60)
print(f"Total parameters     : {total_params:>12,}")
print(f"Trainable parameters : {trainable_params:>12,}")
print(f"Device               : {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Swin Transformer — Base
Total parameters     :   86,745,274
Trainable parameters :   86,745,274
Device               : cuda


In [47]:
print("\nSW-ViT Layer-by-Layer Breakdown")
print("=" * 60)
for name, module in model_swvit.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:<25} {n_params:>12,} params")

print(f"\nInput layer  : patch_embed projects 3-channel image into 128-d tokens")
print(f"Output layer : head Linear(1024, 2) + Softmax for binary classification")


SW-ViT Layer-by-Layer Breakdown
patch_embed                      6,528 params
layers                      86,734,648 params
norm                             2,048 params
head                             2,050 params

Input layer  : patch_embed projects 3-channel image into 128-d tokens
Output layer : head Linear(1024, 2) + Softmax for binary classification


In [46]:
print("\nSW-ViT Saving and Loading")
print("=" * 60)

save_path = "weights/swvit_demo.pth"
torch.save(model_swvit.state_dict(), save_path)
print(f"Saved state_dict to: {save_path}  ({os.path.getsize(save_path)/1e6:.1f} MB)")

# Reload
model_loaded = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=2)
model_loaded.load_state_dict(torch.load(save_path, map_location="cpu", weights_only=True))
model_loaded.eval()
print("Reloaded successfully model in eval mode")


SW-ViT Saving and Loading
Saved state_dict to: weights/swvit_demo.pth  (347.1 MB)
Reloaded successfully model in eval mode


## 4. Model Comparison Summary

| | YOLOv8n | YOLO12n | SW-ViT (Base) |
|---|---------|---------|---------------|
| **Task** | Object detection | Object detection | Image classification |
| **Parameters** | ~3.0 M | ~2.6 M | ~86.7 M |
| **Input** | 640 × 640 × 3 | 640 × 640 × 3 | 224 × 224 × 3 |
| **Output** | Bboxes + class | Bboxes + class | Class probability |
| **Hidden act.** | SiLU | SiLU | GELU |
| **Output act.** | Sigmoid | Sigmoid | Softmax |
| **Loss** | CIoU + BCE + DFL | CIoU + BCE + DFL | CrossEntropy |